In [1]:
# ============================================================
# Patch extraction pipeline - resume / memory safe version
# ------------------------------------------------------------
# 목적:
#   - 전처리 완료 결과에서 patch 추출
#   - 환자별 patch CSV로 바로 저장
#   - 이미 처리된 환자는 자동 skip
#   - 전체 patch를 메모리에 한 번에 쌓지 않음
#   - MemoryError 방지
# ============================================================

from pathlib import Path
import json
import gc
import time
from collections import Counter

import numpy as np
import pandas as pd
import SimpleITK as sitk
from tqdm import tqdm
from scipy.ndimage import distance_transform_edt


# ============================================================
# 1. CONFIG
# ============================================================

PATCH_CONFIG = {
    # 전체 전처리 결과 폴더
    "preprocess_root": r"E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v2_tslungguard_nochest",

    # 패치 결과 저장 폴더 이름
    "out_dir_name": "patch_all_v2_tslungguard_nochest_resume",

    # patch 크기
    "patch_size": 32,

    # patch 이동 간격
    # 32 크기 patch를 16칸씩 이동하므로 반씩 겹침
    "patch_stride": 16,

    # slice 자체에 pure lung이 거의 없으면 제외
    "min_slice_pure_lung_ratio": 0.01,

    # 너무 작은 폐 mask slice 제외
    "min_pure_lung_pixels_per_slice": 100,

    # patch 안 pure lung 비율
    # 0.70이면 patch 안 70% 이상이 pure lung이어야 선택
    "min_patch_pure_lung_ratio": 0.70,

    # patch 안 장기 제외 영역이 2% 넘으면 제외
    "max_patch_organ_ratio": 0.02,

    # 중심부 / 말초부 기준
    # 폐 경계에서 가장 먼 거리 대비 평균 0.5 이상이면 central
    "central_distance_ratio_threshold": 0.50,

    # HU histogram bin
    "hist_bins": [
        -1000, -900, -800, -700, -600, -500, -400,
        -300, -200, -100, 0, 100, 200, 400
    ],

    # 이미 처리된 환자 CSV가 있으면 skip
    "skip_existing_patient_csv": True,

    # feature 계산 기준
    # False: 기존 방식 유지, patch 전체 HU로 feature 계산
    # True : patch 안 pure_lung 픽셀만 골라 feature 계산
    "feature_use_pure_lung_pixels_only": True,
}


PRE_ROOT = Path(PATCH_CONFIG["preprocess_root"])
PATCH_OUT = PRE_ROOT / PATCH_CONFIG["out_dir_name"]
PATCH_BY_PATIENT_DIR = PATCH_OUT / "patches_by_patient"
PATCH_DONE_DIR = PATCH_OUT / "done_markers"

PATCH_OUT.mkdir(parents=True, exist_ok=True)
PATCH_BY_PATIENT_DIR.mkdir(parents=True, exist_ok=True)
PATCH_DONE_DIR.mkdir(parents=True, exist_ok=True)

with open(PATCH_OUT / "patch_config.json", "w", encoding="utf-8") as f:
    json.dump(PATCH_CONFIG, f, indent=2, ensure_ascii=False)

print("========== PATCH CONFIG ==========")
print("PRE_ROOT:", PRE_ROOT)
print("PATCH_OUT:", PATCH_OUT)
print("PATCH_BY_PATIENT_DIR:", PATCH_BY_PATIENT_DIR)


# ============================================================
# 2. helper functions
# ============================================================

def format_seconds(seconds: float) -> str:
    seconds = int(seconds)

    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h}시간 {m}분 {s}초"

    if m > 0:
        return f"{m}분 {s}초"

    return f"{s}초"


def read_nii_array(path: Path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    return img, arr


def safe_patient_csv_name(patient_id: str) -> str:
    return (
        str(patient_id)
        .replace("\\", "_")
        .replace("/", "_")
        .replace(":", "_")
        .replace("*", "_")
        .replace("?", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
    )


def calc_patch_features(patch_hu, hist_bins):
    """
    patch HU 값으로 간단 feature 계산.
    patch_hu는 2D patch이거나, pure_lung 픽셀만 골라낸 1D 배열일 수 있음.
    """

    x = patch_hu.reshape(-1)

    if x.size == 0:
        feat = {
            "hu_mean": np.nan,
            "hu_std": np.nan,
            "hu_min": np.nan,
            "hu_max": np.nan,
            "hu_p05": np.nan,
            "hu_p25": np.nan,
            "hu_p50": np.nan,
            "hu_p75": np.nan,
            "hu_p95": np.nan,
        }

        for i in range(len(hist_bins) - 1):
            feat[f"hist_bin_{i:02d}"] = np.nan

        return feat

    x = x.astype(np.float32, copy=False)

    hist, _ = np.histogram(x, bins=hist_bins)
    hist = hist.astype(np.float32)

    hist_sum = hist.sum()

    if hist_sum > 0:
        hist = hist / hist_sum

    feat = {
        "hu_mean": float(np.mean(x)),
        "hu_std": float(np.std(x)),
        "hu_min": float(np.min(x)),
        "hu_max": float(np.max(x)),
        "hu_p05": float(np.percentile(x, 5)),
        "hu_p25": float(np.percentile(x, 25)),
        "hu_p50": float(np.percentile(x, 50)),
        "hu_p75": float(np.percentile(x, 75)),
        "hu_p95": float(np.percentile(x, 95)),
    }

    for i, v in enumerate(hist):
        feat[f"hist_bin_{i:02d}"] = float(v)

    return feat


def get_z_level(z, valid_z_min, valid_z_max):
    """
    lung range 안에서 z 위치를 lower / middle / upper로 나눔.
    """

    if valid_z_max <= valid_z_min:
        return "middle", 0.5

    z_ratio = (z - valid_z_min) / float(valid_z_max - valid_z_min)
    z_ratio = float(np.clip(z_ratio, 0.0, 1.0))

    if z_ratio < 1.0 / 3.0:
        return "lower", z_ratio

    if z_ratio < 2.0 / 3.0:
        return "middle", z_ratio

    return "upper", z_ratio


def get_central_peripheral(dist_patch_values, threshold=0.5):
    """
    patch 안 pure_lung 픽셀들의 distance ratio 평균으로 central/peripheral 결정.
    """

    if len(dist_patch_values) == 0:
        return "peripheral", 0.0

    mean_ratio = float(np.mean(dist_patch_values))

    if mean_ratio >= threshold:
        return "central", mean_ratio

    return "peripheral", mean_ratio


def get_left_right_by_patch_center(x0, patch_size, width):
    """
    좌우 정보는 일단 metadata로만 저장.
    image_left / image_right는 영상 좌표 기준.
    """

    cx = x0 + patch_size / 2.0

    if cx < width / 2.0:
        return "image_left"

    return "image_right"


# ============================================================
# 3. one patient patch extraction
# ============================================================

def extract_patches_one_patient(patient_id, row0, config):
    patch_size = int(config["patch_size"])
    patch_stride = int(config["patch_stride"])
    patch_area = patch_size * patch_size

    ct_path = Path(row0["ct_1mm_lung_range_nii"])
    pure_path = Path(row0["pure_lung_lung_range_nii"])
    organ_path = Path(row0["organ_exclusion_lung_range_nii"])

    if not ct_path.exists():
        raise FileNotFoundError(f"CT lung range 없음: {ct_path}")

    if not pure_path.exists():
        raise FileNotFoundError(f"pure lung range 없음: {pure_path}")

    if not organ_path.exists():
        raise FileNotFoundError(f"organ exclusion range 없음: {organ_path}")

    _, ct_arr = read_nii_array(ct_path)
    _, pure_arr = read_nii_array(pure_path)
    _, organ_arr = read_nii_array(organ_path)

    # 중요:
    # ct_arr는 전체 float32 복사하지 않음.
    # feature 계산할 때 patch 단위로만 float32 변환함.
    pure_arr = pure_arr > 0
    organ_arr = organ_arr > 0

    if ct_arr.shape != pure_arr.shape:
        raise RuntimeError(f"{patient_id}: CT와 pure_lung shape 다름")

    if ct_arr.shape != organ_arr.shape:
        raise RuntimeError(f"{patient_id}: CT와 organ_exclusion shape 다름")

    zdim, h, w = ct_arr.shape

    pure_area_per_slice = pure_arr.sum(axis=(1, 2))
    pure_ratio_per_slice = pure_area_per_slice / float(h * w)

    valid_z_indices = np.where(
        (pure_ratio_per_slice >= float(config["min_slice_pure_lung_ratio"]))
        & (pure_area_per_slice >= int(config["min_pure_lung_pixels_per_slice"]))
    )[0]

    if len(valid_z_indices) == 0:
        summary = {
            "patient_id": patient_id,
            "status": "no_valid_z",
            "zdim": int(zdim),
            "height": int(h),
            "width": int(w),
            "valid_z_min": "",
            "valid_z_max": "",
            "valid_z_count": 0,
            "candidate_patch_count_before_filter": 0,
            "selected_patch_count": 0,
        }

        del ct_arr, pure_arr, organ_arr
        gc.collect()

        return [], summary

    valid_z_min = int(valid_z_indices.min())
    valid_z_max = int(valid_z_indices.max())

    patch_rows = []
    patient_candidate_count = 0
    patient_selected_count = 0

    for z in valid_z_indices:
        z = int(z)

        ct_slice = ct_arr[z]
        pure_slice = pure_arr[z]
        organ_slice = organ_arr[z]

        dist_map = distance_transform_edt(pure_slice)
        max_dist = float(dist_map.max())

        if max_dist <= 0:
            continue

        dist_ratio_map = dist_map / (max_dist + 1e-6)

        z_level, z_ratio = get_z_level(
            z=z,
            valid_z_min=valid_z_min,
            valid_z_max=valid_z_max,
        )

        for y0 in range(0, h - patch_size + 1, patch_stride):
            for x0 in range(0, w - patch_size + 1, patch_stride):
                patient_candidate_count += 1

                y1 = y0 + patch_size
                x1 = x0 + patch_size

                pure_patch = pure_slice[y0:y1, x0:x1]
                organ_patch = organ_slice[y0:y1, x0:x1]

                pure_pixels = int(pure_patch.sum())
                organ_pixels = int(organ_patch.sum())

                pure_ratio = pure_pixels / float(patch_area)
                organ_ratio = organ_pixels / float(patch_area)

                if pure_ratio < float(config["min_patch_pure_lung_ratio"]):
                    continue

                if organ_ratio > float(config["max_patch_organ_ratio"]):
                    continue

                ct_patch = ct_slice[y0:y1, x0:x1]

                dist_patch = dist_ratio_map[y0:y1, x0:x1]
                dist_values_inside_pure = dist_patch[pure_patch > 0]

                central_bin, central_ratio_mean = get_central_peripheral(
                    dist_patch_values=dist_values_inside_pure,
                    threshold=float(config["central_distance_ratio_threshold"]),
                )

                position_bin = f"{z_level}_{central_bin}"

                left_right_bin = get_left_right_by_patch_center(
                    x0=x0,
                    patch_size=patch_size,
                    width=w,
                )

                if bool(config["feature_use_pure_lung_pixels_only"]):
                    feature_pixels = ct_patch[pure_patch > 0]
                else:
                    feature_pixels = ct_patch

                features = calc_patch_features(
                    patch_hu=feature_pixels,
                    hist_bins=config["hist_bins"],
                )

                row = {
                    "patient_id": patient_id,

                    "local_z": int(z),
                    "y0": int(y0),
                    "x0": int(x0),
                    "y1": int(y1),
                    "x1": int(x1),

                    "patch_size": int(patch_size),
                    "patch_stride": int(patch_stride),

                    "pure_lung_pixels": int(pure_pixels),
                    "organ_pixels": int(organ_pixels),
                    "pure_lung_patch_ratio": float(pure_ratio),
                    "organ_patch_ratio": float(organ_ratio),
                    "slice_pure_lung_ratio": float(pure_ratio_per_slice[z]),

                    "z_level": z_level,
                    "z_ratio": float(z_ratio),
                    "central_peripheral": central_bin,
                    "central_distance_ratio_mean": float(central_ratio_mean),
                    "position_bin": position_bin,

                    "left_right_metadata": left_right_bin,

                    "ct_1mm_lung_range_nii": str(ct_path),
                    "pure_lung_lung_range_nii": str(pure_path),
                    "organ_exclusion_lung_range_nii": str(organ_path),
                }

                row.update(features)

                patch_rows.append(row)
                patient_selected_count += 1

    summary = {
        "patient_id": patient_id,
        "status": "success",
        "zdim": int(zdim),
        "height": int(h),
        "width": int(w),
        "valid_z_min": int(valid_z_min),
        "valid_z_max": int(valid_z_max),
        "valid_z_count": int(len(valid_z_indices)),
        "candidate_patch_count_before_filter": int(patient_candidate_count),
        "selected_patch_count": int(patient_selected_count),
    }

    del ct_arr, pure_arr, organ_arr
    gc.collect()

    return patch_rows, summary


# ============================================================
# 4. run patch extraction with resume
# ============================================================

metadata_path = PRE_ROOT / "metadata_slices.csv"

if not metadata_path.exists():
    raise FileNotFoundError(f"metadata_slices.csv 없음: {metadata_path}")

slice_df = pd.read_csv(metadata_path)

required_cols = [
    "patient_id",
    "ct_1mm_lung_range_nii",
    "pure_lung_lung_range_nii",
    "organ_exclusion_lung_range_nii",
]

for c in required_cols:
    if c not in slice_df.columns:
        raise RuntimeError(f"metadata_slices.csv에 필요한 컬럼 없음: {c}")

patient_ids = sorted(slice_df["patient_id"].unique())

print("========== PATCH EXTRACTION START ==========")
print("patient count:", len(patient_ids))
print("PATCH_OUT:", PATCH_OUT)

summary_rows = []
error_rows = []

global_start = time.perf_counter()
total_cases = len(patient_ids)

for case_idx, patient_id in enumerate(
    tqdm(
        patient_ids,
        desc="Patch extraction by patient",
        total=total_cases,
        ncols=100,
        ascii=True,
        leave=True,
        dynamic_ncols=False,
    ),
    start=1
):
    start = time.perf_counter()

    patient_safe = safe_patient_csv_name(patient_id)
    patient_csv = PATCH_BY_PATIENT_DIR / f"{patient_safe}.csv"
    patient_tmp_csv = PATCH_BY_PATIENT_DIR / f"{patient_safe}.tmp.csv"
    done_marker = PATCH_DONE_DIR / f"{patient_safe}.done"

    if (
        bool(PATCH_CONFIG["skip_existing_patient_csv"])
        and patient_csv.exists()
        and done_marker.exists()
    ):
        elapsed = time.perf_counter() - start

        summary_rows.append({
            "patient_id": patient_id,
            "status": "skipped_existing",
            "patient_csv": str(patient_csv),
            "elapsed_seconds": round(elapsed, 2),
            "elapsed_readable": format_seconds(elapsed),
        })

        continue

    try:
        pdf = slice_df[slice_df["patient_id"] == patient_id]

        if len(pdf) == 0:
            raise RuntimeError(f"{patient_id}: metadata_slices.csv에 환자 row 없음")

        row0 = pdf.iloc[0]

        patch_rows, summary = extract_patches_one_patient(
            patient_id=patient_id,
            row0=row0,
            config=PATCH_CONFIG,
        )

        patient_df = pd.DataFrame(patch_rows)

        # tmp 파일로 먼저 저장하고, 성공하면 최종 파일명으로 교체
        patient_df.to_csv(patient_tmp_csv, index=False, encoding="utf-8-sig")
        patient_tmp_csv.replace(patient_csv)

        done_marker.write_text("done", encoding="utf-8")

        elapsed = time.perf_counter() - start
        total_elapsed = time.perf_counter() - global_start
        avg_per_case = total_elapsed / case_idx
        remain = total_cases - case_idx
        eta = avg_per_case * remain

        summary.update({
            "patient_csv": str(patient_csv),
            "elapsed_seconds": round(elapsed, 2),
            "elapsed_readable": format_seconds(elapsed),
            "total_elapsed_seconds": round(total_elapsed, 2),
            "total_elapsed_readable": format_seconds(total_elapsed),
            "estimated_remaining_seconds": round(eta, 2),
            "estimated_remaining_readable": format_seconds(eta),
        })

        summary_rows.append(summary)

        pd.DataFrame(summary_rows).to_csv(
            PATCH_OUT / "patch_patient_summary.csv",
            index=False,
            encoding="utf-8-sig",
        )

        print(
            f"[PATCH] {patient_id} 완료 "
            f"({case_idx}/{total_cases}) | "
            f"patch={len(patient_df)} | "
            f"이번 환자: {format_seconds(elapsed)} | "
            f"전체 경과: {format_seconds(total_elapsed)} | "
            f"예상 남은 시간: {format_seconds(eta)}"
        )

        del patch_rows, patient_df
        gc.collect()

    except Exception as e:
        elapsed = time.perf_counter() - start

        error_rows.append({
            "patient_id": patient_id,
            "error": str(e),
            "elapsed_seconds": round(elapsed, 2),
            "elapsed_readable": format_seconds(elapsed),
        })

        pd.DataFrame(error_rows).to_csv(
            PATCH_OUT / "patch_errors.csv",
            index=False,
            encoding="utf-8-sig",
        )

        print("[ERROR]", patient_id, str(e))
        gc.collect()


summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(
    PATCH_OUT / "patch_patient_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

if len(error_rows) > 0:
    pd.DataFrame(error_rows).to_csv(
        PATCH_OUT / "patch_errors.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("\n========== PATCH EXTRACTION FINISHED ==========")
print("PATCH_OUT:", PATCH_OUT)
print("patient csv dir:", PATCH_BY_PATIENT_DIR)
print("summary:", PATCH_OUT / "patch_patient_summary.csv")
print("errors:", len(error_rows))


# ============================================================
# 5. position_bin count summary
# ------------------------------------------------------------
# 전체 patch를 한 번에 메모리에 올리지 않고,
# 환자별 CSV를 하나씩 읽어서 position count만 집계.
# ============================================================

position_counter = Counter()
patient_patch_counts = []

patient_csv_files = sorted(PATCH_BY_PATIENT_DIR.glob("*.csv"))

for csv_path in tqdm(
    patient_csv_files,
    desc="Summarize patch CSVs",
    total=len(patient_csv_files),
    ncols=100,
    ascii=True,
    leave=True,
    dynamic_ncols=False,
):
    df = pd.read_csv(csv_path, usecols=["patient_id", "position_bin"])

    if len(df) == 0:
        continue

    patient_id = df["patient_id"].iloc[0]
    counts = df["position_bin"].value_counts()

    for k, v in counts.items():
        position_counter[k] += int(v)

    patient_patch_counts.append({
        "patient_id": patient_id,
        "patch_count": int(len(df)),
    })

position_df = pd.DataFrame([
    {"position_bin": k, "patch_count": v}
    for k, v in sorted(position_counter.items())
])

patient_count_df = pd.DataFrame(patient_patch_counts)

position_df.to_csv(
    PATCH_OUT / "patch_position_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

patient_count_df.to_csv(
    PATCH_OUT / "patch_count_by_patient.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n========== Position bin counts ==========")
display(position_df)

print("\n========== Patch count by patient summary ==========")

if len(patient_count_df) > 0:
    display(patient_count_df["patch_count"].describe())
else:
    print("patient_count_df 비어 있음")

print("\nSaved:")
print(PATCH_OUT / "patch_position_counts.csv")
print(PATCH_OUT / "patch_count_by_patient.csv")

========== PATCH CONFIG ==========
PRE_ROOT: E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v2_tslungguard_nochest
PATCH_OUT: E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v2_tslungguard_nochest\patch_all_v2_tslungguard_nochest_resume
PATCH_BY_PATIENT_DIR: E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v2_tslungguard_nochest\patch_all_v2_tslungguard_nochest_resume\patches_by_patient
========== PATCH EXTRACTION START ==========
patient count: 362
PATCH_OUT: E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v2_tslungguard_nochest\patch_all_v2_tslungguard_nochest_resume


Patch extraction by patient:   0%|                                | 1/362 [00:19<1:56:39, 19.39s/it]

[PATCH] normal001 완료 (1/362) | patch=20948 | 이번 환자: 19초 | 전체 경과: 19초 | 예상 남은 시간: 1시간 56분 17초


Patch extraction by patient:   1%|1                               | 2/362 [00:39<1:58:10, 19.69s/it]

[PATCH] normal002 완료 (2/362) | patch=22012 | 이번 환자: 19초 | 전체 경과: 39초 | 예상 남은 시간: 1시간 57분 42초


...


========== Position bin counts ==========


,position_bin,patch_count
0,lower_central,838632
1,lower_peripheral,2188356
2,middle_central,1721819
3,middle_peripheral,3787892
4,upper_central,995664
5,upper_peripheral,2146744



========== Patch count by patient summary ==========


count      362.000000
mean     32262.726519
std       9847.895441
min       7594.000000
25%      25986.750000
50%      32488.000000
75%      38226.250000
max      66124.000000
Name: patch_count, dtype: float64


Saved:
E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v2_tslungguard_nochest\patch_all_v2_tslungguard_nochest_resume\patch_position_counts.csv
E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v2_tslungguard_nochest\patch_all_v2_tslungguard_nochest_resume\patch_count_by_patient.csv


In [1]:
from pathlib import Path
import pandas as pd

PATCH_OUT = Path(r"E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v1\patch_all_v2_resume")
PATCH_BY_PATIENT_DIR = PATCH_OUT / "patches_by_patient"
PATCH_DONE_DIR = PATCH_OUT / "done_markers"

csv_files = sorted([p for p in PATCH_BY_PATIENT_DIR.glob("*.csv") if not p.name.endswith(".tmp.csv")])
tmp_files = sorted(PATCH_BY_PATIENT_DIR.glob("*.tmp.csv"))
done_files = sorted(PATCH_DONE_DIR.glob("*.done"))

print("patient csv count:", len(csv_files))
print("tmp csv count:", len(tmp_files))
print("done marker count:", len(done_files))

summary_df = pd.read_csv(PATCH_OUT / "patch_patient_summary.csv")
count_df = pd.read_csv(PATCH_OUT / "patch_count_by_patient.csv")
position_df = pd.read_csv(PATCH_OUT / "patch_position_counts.csv")

print("summary rows:", len(summary_df))
print("patch_count_by_patient rows:", len(count_df))
print("position rows:", len(position_df))

print("\nsummary status counts:")
print(summary_df["status"].value_counts(dropna=False))

print("\nposition counts:")
display(position_df)

print("\npatch count summary:")
display(count_df["patch_count"].describe())

patient csv count: 362
tmp csv count: 0
done marker count: 362
summary rows: 362
patch_count_by_patient rows: 362
position rows: 6

summary status counts:
status
success    362
Name: count, dtype: int64

position counts:


,position_bin,patch_count
0,lower_central,829688
1,lower_peripheral,2214954
2,middle_central,1815279
3,middle_peripheral,4128678
4,upper_central,998059
5,upper_peripheral,2246813



patch count summary:


count      362.000000
mean     33794.118785
std       9955.532283
min      10362.000000
25%      27313.000000
50%      33391.500000
75%      39496.250000
max      73093.000000
Name: patch_count, dtype: float64